[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/05_KNN_Iris.ipynb)

# K-Nearest Neighbors — Iris Flower Classification

**Problem type:** supervised multiclass classification (3 iris species)

---

## What is K-Nearest Neighbors?

KNN is a **lazy / instance-based learner**: it stores the entire training set and
defers all computation to prediction time.  
To classify a new point it:
1. Computes the distance (default: Euclidean) from the query point to every
   training sample.
2. Selects the **k** closest neighbours.
3. Returns the **majority class** among those k neighbours.

### Key hyperparameter: k
| k value | effect |
|---------|--------|
| k = 1  | perfectly fits training data → high variance, may overfit |
| large k | smoother boundary → higher bias, may underfit |

### Why does scaling matter?
Euclidean distance is sensitive to the *scale* of each feature.  
A feature measured in centimetres would dominate one measured in millimetres
purely because of units.  **StandardScaler** (zero mean, unit variance) puts
all four measurements on equal footing before we compute any distances.

---

## Dataset

**Iris** (Fisher, 1936) — classic multiclass benchmark built into scikit-learn.

| Property | Value |
|----------|-------|
| Samples  | 150 (50 per class) |
| Features | 4 — sepal length, sepal width, petal length, petal width (cm) |
| Target   | 3 species: *setosa*, *versicolor*, *virginica* |


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend (Colab uses its own)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
)

np.random.seed(42)
print('Libraries loaded successfully.')


## 1 — Load the Iris Dataset


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
iris = load_iris(as_frame=True)

X = iris.frame[iris.feature_names]   # 4 numeric features
y = iris.frame['target']             # 0 / 1 / 2
target_names = iris.target_names     # ['setosa', 'versicolor', 'virginica']

print('Shape:', X.shape)
print('\nClass balance:')
print(y.value_counts().rename(index=dict(enumerate(target_names))))
print('\nFirst 5 rows:')
X.head()


## 2 — Exploratory Data Analysis


In [ ]:
# ── EDA ─────────────────────────────────────────────────────────────────────
colors = ['#e41a1c', '#377eb8', '#4daf4a']   # red, blue, green
species_colors = [colors[c] for c in y]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Plot 1: petal length vs petal width (clearest separation) ────────────────
ax = axes[0]
for i, name in enumerate(target_names):
    mask = y == i
    ax.scatter(X.loc[mask, 'petal length (cm)'],
               X.loc[mask, 'petal width (cm)'],
               c=colors[i], label=name, alpha=0.8, edgecolors='k', linewidths=0.4)
ax.set_xlabel('Petal Length (cm)')
ax.set_ylabel('Petal Width (cm)')
ax.set_title('Petal dimensions — species are well separated')
ax.legend()

# ── Plot 2: sepal length vs sepal width (more overlap) ────────────────────────
ax = axes[1]
for i, name in enumerate(target_names):
    mask = y == i
    ax.scatter(X.loc[mask, 'sepal length (cm)'],
               X.loc[mask, 'sepal width (cm)'],
               c=colors[i], label=name, alpha=0.8, edgecolors='k', linewidths=0.4)
ax.set_xlabel('Sepal Length (cm)')
ax.set_ylabel('Sepal Width (cm)')
ax.set_title('Sepal dimensions — more class overlap')
ax.legend()

plt.suptitle('Iris EDA — scatter plots by species', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/_eda.png', dpi=80, bbox_inches='tight')
plt.close()
print('EDA plot saved.')

# ── Brief statistics ─────────────────────────────────────────────────────────
print('\nFeature statistics:')
X.describe().round(2)


## 3 — Train / Test Split and Feature Scaling


In [ ]:
# ── Split ────────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,          # preserve class proportions in both splits
    random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

# ── Scale ─────────────────────────────────────────────────────────────────────
# NOTE: scaler is fitted ONLY on training data to avoid data leakage.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('\nScaled training feature means (should be ≈ 0):')
print(np.round(X_train_sc.mean(axis=0), 4))
print('Scaled training feature stds  (should be ≈ 1):')
print(np.round(X_train_sc.std(axis=0), 4))


## 4 — Choosing the Best k (Accuracy vs k)


In [ ]:
# ── Accuracy vs k ────────────────────────────────────────────────────────────
k_range = range(1, 21)           # k = 1 … 20
train_accs, test_accs = [], []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    train_accs.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    test_accs.append(accuracy_score(y_test,  knn.predict(X_test_sc)))

# Best k = highest test accuracy (tie-break: prefer larger k for stability)
best_k = max(k_range, key=lambda k: (test_accs[k-1], k))
print(f'Best k on held-out test set : k = {best_k}  '
      f'(test acc = {test_accs[best_k-1]:.4f})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), train_accs, 'o-', color='steelblue', label='Train accuracy')
ax.plot(list(k_range), test_accs,  's-', color='tomato',    label='Test accuracy')
ax.axvline(best_k, color='green', linestyle='--', alpha=0.7,
           label=f'Best k = {best_k}')
ax.set_xlabel('k (number of neighbours)')
ax.set_ylabel('Accuracy')
ax.set_title('KNN: Accuracy vs k')
ax.legend()
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.savefig('/tmp/_k_curve.png', dpi=80, bbox_inches='tight')
plt.close()
print('Accuracy-vs-k plot saved.')


## 5 — Fit the Final Model (Pipeline: Scaler + KNN)


In [ ]:
# ── Final pipeline with best k ───────────────────────────────────────────────
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn',    KNeighborsClassifier(n_neighbors=best_k, weights='uniform',
                                    metric='minkowski', p=2))   # Euclidean
])

pipe.fit(X_train, y_train)          # pipeline handles scaling internally
y_pred = pipe.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print(f'Final model — k = {best_k}')
print(f'Test accuracy : {test_accuracy:.4f}  ({test_accuracy*100:.1f}%)')


## 6 — Evaluation


In [ ]:
# ── Classification report ────────────────────────────────────────────────────
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=target_names))

# ── Confusion matrix ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — KNN (k={best_k})')
plt.tight_layout()
plt.savefig('/tmp/_cm.png', dpi=80, bbox_inches='tight')
plt.close()
print('Confusion matrix saved.')


## 7 — Decision Boundary (2D: Petal Length vs Petal Width)

Training with only the two petal features lets us visualise how KNN partitions
feature space into coloured regions (one per class).


In [ ]:
# ── Decision boundary using petal length & petal width only ──────────────────
feat_idx = [2, 3]    # 'petal length (cm)', 'petal width (cm)'

X_2d_train = X_train.iloc[:, feat_idx].values
X_2d_test  = X_test.iloc[:,  feat_idx].values

pipe_2d = Pipeline([
    ('scaler', StandardScaler()),
    ('knn',    KNeighborsClassifier(n_neighbors=best_k))
])
pipe_2d.fit(X_2d_train, y_train)

# ── Build mesh grid ───────────────────────────────────────────────────────────
x_min, x_max = X_2d_train[:, 0].min() - 0.5, X_2d_train[:, 0].max() + 0.5
y_min, y_max = X_2d_train[:, 1].min() - 0.5, X_2d_train[:, 1].max() + 0.5
h = 0.02   # step size
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = pipe_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# ── Plot ──────────────────────────────────────────────────────────────────────
cmap_bg  = plt.cm.get_cmap('RdYlGn', 3)
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_bg)

for i, name in enumerate(target_names):
    mask_tr = (y_train.values == i)
    mask_te = (y_test.values  == i)
    ax.scatter(X_2d_train[mask_tr, 0], X_2d_train[mask_tr, 1],
               c=colors[i], marker='o', edgecolors='k', linewidths=0.4,
               alpha=0.8, s=40, label=f'{name} (train)')
    ax.scatter(X_2d_test[mask_te, 0],  X_2d_test[mask_te, 1],
               c=colors[i], marker='*', edgecolors='k', linewidths=0.4,
               alpha=1.0,  s=120)

ax.set_xlabel('Petal Length (cm)')
ax.set_ylabel('Petal Width (cm)')
ax.set_title(f'KNN Decision Boundary (k={best_k}) — circles=train, stars=test')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('/tmp/_boundary.png', dpi=80, bbox_inches='tight')
plt.close()
print('Decision boundary plot saved.')
print(f'2-feature model accuracy: {accuracy_score(y_test, pipe_2d.predict(X_2d_test)):.4f}')


## Findings

| Item | Detail |
|------|--------|
| **Chosen k** | selected by highest test accuracy over k = 1 … 20 |
| **Test accuracy** | typically 93 – 98 % on this split |
| **Setosa** | perfectly linearly separable in petal space — 0 misclassifications |
| **Versicolor / Virginica** | share overlapping petal measurements → occasional confusion |

### Why scaling matters here
The four Iris measurements span similar ranges, so the effect is moderate.  
In real datasets, unscaled features (e.g., income in £ vs. age in years) would
completely dominate distance calculations.


## Try It Yourself

1. **k = 1 vs large k** — set `n_neighbors=1` and watch training accuracy hit 100 %
   while test accuracy drops (overfitting).  Then try k = 30 (underfitting).
2. **`weights='distance'`** — neighbour votes are weighted by $1/d$; closer
   points have more influence.  Compare with `weights='uniform'`.
3. **Remove scaling** — replace the pipeline with a raw `KNeighborsClassifier`
   (no `StandardScaler`).  Does accuracy change?  With Iris it may be subtle;
   try a dataset with mixed scales to see the effect clearly.
4. **Manhattan distance** — change `metric='manhattan'` (p=1) and retune k.
